In [ ]:
import os
import tqdm
import numpy as np
import pandas as pd
from tushare import pro_api, set_token
from datetime import datetime, timedelta
from datatools import get_price


set_token('b549c6e18f71105519447d872727cc2b7f0022f071c9eb27d0256ebb')
pro = pro_api()

In [ ]:
# 交易日历
trade_cal = pro.trade_cal()
data_cal = trade_cal[(trade_cal['cal_date']>='20140101')&(trade_cal['cal_date']<'20260101')]
trade_cal.to_csv('data/trade_cal.csv', index=False)

## 获取行情数据

In [ ]:
data_path = 'data/daily_K/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.daily(trade_date=d['cal_date'])
        adj_factor = pro.adj_factor(trade_date=d['cal_date'])
        price = pd.merge(daily, adj_factor)
        price = price.drop(columns=['pre_close', 'change', 'pct_chg']).sort_values('ts_code').reset_index(drop=True)
        price['trade_date'] = pd.to_datetime(price['trade_date'])
        price.to_feather(os.path.join(data_path, f'stock-{d["cal_date"]}.ftr'))
    else:
        continue

4383it [31:13,  2.34it/s]


In [24]:
datas = get_price(codes = ['000001.SZ'], start_date='2023-03-01', end_date='2023-07-17', fq=None)

### 获取每日指标

In [36]:
data_path = 'data/daily_basic/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily_basic = pro.daily_basic(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        daily_basic['trade_date'] = pd.to_datetime(daily_basic['trade_date'])
        daily_basic.to_feather(os.path.join(data_path, f'basic-{d["cal_date"]}.ftr'))
    else:
        continue

4383it [24:48,  2.95it/s]
